In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "XRPUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from optuna.pruners import MedianPruner
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 284,679


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,is_trending,hour,hour_sin,hour_cos,dow_sin,dow_cos,dom_sin,dom_cos,month_sin,month_cos
0,2025-09-01 00:00:00+00:00,2.7757,2.7757,2.7723,2.7757,83483.8,2025-09-01 00:00:59.999999+00:00,2.315646e+05,1015,58143.3,...,0,0,0.0,1.0,0.0,1.0,0.201299,0.97953,-1.0,-1.836970e-16
1,2025-09-01 00:01:00+00:00,2.7757,2.7772,2.7753,2.7772,84999.6,2025-09-01 00:01:59.999999+00:00,2.359481e+05,696,62509.2,...,0,0,0.0,1.0,0.0,1.0,0.201299,0.97953,-1.0,-1.836970e-16
2,2025-09-01 00:02:00+00:00,2.7772,2.7774,2.7756,2.7767,36682.2,2025-09-01 00:02:59.999999+00:00,1.018428e+05,582,21315.5,...,0,0,0.0,1.0,0.0,1.0,0.201299,0.97953,-1.0,-1.836970e-16
3,2025-09-01 00:03:00+00:00,2.7766,2.7770,2.7751,2.7751,28222.0,2025-09-01 00:03:59.999999+00:00,7.834349e+04,843,11713.8,...,0,0,0.0,1.0,0.0,1.0,0.201299,0.97953,-1.0,-1.836970e-16
4,2025-09-01 00:04:00+00:00,2.7751,2.7751,2.7605,2.7629,838170.8,2025-09-01 00:04:59.999999+00:00,2.318988e+06,4641,203901.1,...,0,0,0.0,1.0,0.0,1.0,0.201299,0.97953,-1.0,-1.836970e-16


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 284,601
[info] optuna train rows: 182,144
[info] valid rows:        45,536
[info] test rows:         56,921


In [9]:
pruner = MedianPruner(n_warmup_steps=5, n_min_trials=10)
study = optuna.create_study(direction="maximize", pruner=pruner)

class EarlyStoppingCallback:
    def __init__(self, patience: int):
        self.patience = patience
        self.best_value = -float('inf')
        self.no_improvement_count = 0

    def __call__(self, study, trial):
        if study.best_value > self.best_value:
            self.best_value = study.best_value
            self.no_improvement_count = 0
        else:
            self.no_improvement_count += 1

        if self.no_improvement_count >= self.patience:
            study.stop()

early_stopping = EarlyStoppingCallback(patience=10)

objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, callbacks=[early_stopping], show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-20 07:00:54,162] A new study created in memory with name: no-name-9f06f125-5ea9-4406-a194-20cf31adf9ca


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:03<?, ?it/s]

Best trial: 0. Best value: -0.000100832:   0%|          | 0/50 [00:03<?, ?it/s]

Best trial: 0. Best value: -0.000100832:   2%|▏         | 1/50 [00:03<03:00,  3.68s/it]

...

Best trial: 15. Best value: 0.0178104:  50%|█████     | 25/50 [02:12<02:17,  5.51s/it]

Best trial: 15. Best value: 0.0178104:  50%|█████     | 25/50 [02:12<02:17,  5.51s/it]

Best trial: 15. Best value: 0.0178104:  52%|█████▏    | 26/50 [02:12<01:53,  4.72s/it]

Best trial: 15. Best value: 0.0178104:  52%|█████▏    | 26/50 [02:12<02:02,  5.10s/it]

[I 2026-03-20 07:03:06,718] Trial 25 finished with value: 0.011032977732636942 and parameters: {'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.1026819003420032, 'subsample': 0.9635054926531635, 'colsample_bytree': 0.551741524556275, 'min_child_weight': 9, 'reg_alpha': 1.3677480771185776e-05, 'reg_lambda': 0.8928107110644626}. Best is trial 15 with value: 0.017810369990408505.

[optuna] best trial
value: 0.017810
params:
  n_estimators: 1800
  max_depth: 7
  learning_rate: 0.19372892072446862
  subsample: 0.8918944822180366
  colsample_bytree: 0.5827714781211446
  min_child_weight: 8
  reg_alpha: 0.026899105763061034
  reg_lambda: 0.21687650885294632


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final xgb...


[training] done in 9.30s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:      0.940242
Test IC:       0.013313
Train Rank IC: 0.844958
Test Rank IC:  0.021217
Train RMSE:    0.001011
Test RMSE:     0.002716


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
trend_strength      0.065765
imbalance_5         0.060467
range_ratio         0.058373
month_sin           0.053549
vol_regime_ratio    0.051145
hour_cos            0.047700
dist_ma_15_z        0.042606
hour_sin            0.040284
dom_sin             0.039997
vol_ratio_5_30      0.037168
imbalance_15        0.036742
dist_ma_15          0.034140
volume_mom_5        0.032566
dom_cos             0.030599
is_trending         0.029334
dow_cos             0.026810
vol_30              0.025570
range_15            0.023646
vol_15              0.023471
mom_3               0.022400
dist_ma_30          0.021883
volume_z            0.021278
dist_ma_5           0.021111
vol_5               0.020988
dow_sin             0.020623
mom_15              0.020528
range_5             0.020192
month_cos           0.019078
mom_10              0.017996
bar_range           0.017323
mom_5               0.016667
dtype: float32


In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/XRPUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/XRPUSDT__h5_model.joblib
[saved] features -> models/xgb/XRPUSDT__h5_feature_cols.json
[saved] feature importance -> models/xgb/XRPUSDT__h5_feature_importance.csv
[saved] metadata -> models/xgb/XRPUSDT__h5_meta.json
